<a href="https://colab.research.google.com/github/Venni2911/LogicMojo-AI-ML-Sept25-VenniRaj/blob/main/Transformer_Implementation_Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch torchtext datasets

In [ ]:
from datasets import load_dataset
dataset = load_dataset('imdb')

train_data = dataset["train"]
test_data = dataset["test"]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
train_data

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [ ]:
### tokenise

### Transform -> tokens -> vectors


from torchtext.data.utils import get_tokenizer

from collections import Counter
from torchtext.vocab import vocab

tokenizer = get_tokenizer("basic_english")

counter = Counter()

for sample in train_data:
  tokens = tokenizer(sample["text"])
  counter.update(tokens)

vocab_obj = vocab(counter, specials = ['<pad>', '<unk>'])
vocab_obj.set_default_index(vocab_obj['<unk>'])

/usr/local/lib/python3.12/dist-packages/torchtext/data/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/usr/local/lib/python3.12/dist-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/usr/local/lib/python3.12/dist-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated

In [ ]:
from IPython.core.displayhook import tokenize
### Text to Sequences

import torch
max_len = 200

def encode(text):
  tokens = tokenizer(text)
  ids = [vocab_obj[token] for token in tokens ]

  if len(ids) < max_len:
    ids += [0]*(max_len - len(ids))
  else:
    ids = ids[:max_len]

  return torch.tensor(ids)

In [ ]:
### Model Implementation


import torch.nn as nn


class TransformerClassifier (nn.Module):

  def __init__(self, vocab_size, embed_dim = 128, num_heads = 4):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embed_dim)
    encoder_layer = nn.TransformerEncoderLayer(
        d_model = embed_dim,
        nhead = num_heads
    )

    self.transformer = nn.TransformerEncoder(
        encoder_layer,
        num_layers = 2
    )

    self.fc = nn.Linear(embed_dim, 2)

  def forward(self, x):
    x = self.embedding(x)
    x = x.permute(1,0,2)
    x = self.transformer(x)
    x = x.mean(dim = 0)
    out = self.fc(x)
    return out


In [ ]:
model = TransformerClassifier(len(vocab_obj))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [ ]:
label = torch.tensor(sample["label"])
label

tensor(0)

In [ ]:
for epoch in range(5):
  total_loss = 0
  for sample in train_data:
    text = encode(sample["text"]).unsqueeze(0)
    label = torch.tensor([sample["label"]])
    output = model(text)
    loss = loss_fn(output, label)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  print("Epoch:", epoch, "Loss:", total_loss)

Epoch: 0 Loss: 45.20030156007471


In [ ]:
## Evals

correct = 0
for sample in test_data :
  text = encode(sample["text"]).unsqueeze(0)
  output = model(text)
  prediction = torch.argmax(output)
  if prediction.item() == sample['label']:
    correct += 1
accuracy = correct/len(test_data)
accuracy

In [ ]:
### predicts

def predict(text):
  encoded = encode(text).unsqueeze(0)
  output = model(encoded)
  label = torch.argmax(output)
  if label == 1
    return "Positive"
  else:
    return "Negative"